## Understanding the transfomer block

### Inspecting Qwen's architecture

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
).to("cuda")

model.eval()

print("Model loaded")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", next(model.parameters()).device)

Model loaded
Parameters: 494032768
Device: cuda:0


In [ ]:
print(model.config) #inspect Qwen's actual architecture 

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "rms_n

In [4]:
layer = model.model.layers[0] #Inspecting the first transformer layer

print(layer)

Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


In [5]:
print("Hidden size:", model.config.hidden_size)
print("Attention heads:", model.config.num_attention_heads)
print("Key-value heads:", model.config.num_key_value_heads)
print("Transformer layers:", model.config.num_hidden_layers)

Hidden size: 896
Attention heads: 14
Key-value heads: 2
Transformer layers: 24


### Manual calculation of self-attention

In [6]:
import torch
import math

# One query, three keys, three values
Q = torch.tensor([[1.0, 0.0]])

K = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

V = torch.tensor([
    [2.0, 0.0],
    [0.0, 3.0],
    [1.0, 1.0]
])

scores = Q @ K.T / math.sqrt(2)
weights = torch.softmax(scores, dim=-1)
output = weights @ V

print("Scores:", scores)
print("Weights:", weights)
print("Attention output:", output)

Scores: tensor([[0.7071, 0.0000, 0.7071]])
Weights: tensor([[0.4011, 0.1978, 0.4011]])
Attention output: tensor([[1.2033, 0.9944]])


In [7]:
import torch

# Attention scores for three query positions and three key positions
scores = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 1.0, 4.0],
    [3.0, 2.0, 1.0]
])

# Mask future positions
mask = torch.triu(
    torch.ones(3, 3, dtype=torch.bool),
    diagonal=1
)

masked_scores = scores.masked_fill(mask, float("-inf"))
attention_weights = torch.softmax(masked_scores, dim=-1)

print("Mask:\n", mask)
print("Masked scores:\n", masked_scores)
print("Attention weights:\n", attention_weights)

Mask:
 tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
Masked scores:
 tensor([[1., -inf, -inf],
        [2., 1., -inf],
        [3., 2., 1.]])
Attention weights:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.7311, 0.2689, 0.0000],
        [0.6652, 0.2447, 0.0900]])
